# Exploring unified medical tokens

Loads real outputs of `train` / `encode` runs (run the pipeline first). Nothing here is synthetic or hard-coded: every cell reads files from `experiments/`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from medical_tokenizer.encode import encode_task
from medical_tokenizer.toolkit import load_tokenizer, build_lexicon, load_text_tokenizer
from medical_tokenizer.protocol import split_spans, to_tokens
lex = build_lexicon()

## 1. A multimodal ICU sequence (templated note + time-series events)

In [ ]:
shards = encode_task('experiments/timeseries_event', 'mortality', include_text=True)
seq = shards['test'][0]
text = load_text_tokenizer()
toks = to_tokens(seq, lex, text.piece)
print(len(seq), 'tokens; labels', seq.labels)
print(' '.join(t.token for t in toks[:80]))

In [ ]:
ts = load_tokenizer('experiments/timeseries_event')
out = ts.decode(split_spans(seq)[1])
plt.imshow(out['mask'].T, aspect='auto', cmap='Greys'); plt.xlabel('hour'); plt.ylabel('variable'); plt.title('observed cells recovered from event tokens')

## 2. Image tokens and reconstruction

In [ ]:
img_tok = load_tokenizer('experiments/image_vq')
img_shards = encode_task('experiments/image_vq', 'bloodmnist')
span = split_spans(img_shards['test'][0])[0]
print(img_tok.codes_from_span(span))
plt.imshow(img_tok.decode(span)); plt.title('decoded from 56 tokens')

## 3. Codebook usage

In [ ]:
c = np.sort(np.load('experiments/image_vq/latest/eval/code_counts.npy'))[::-1]
plt.semilogy(c / c.sum()); plt.xlabel('code rank'); plt.ylabel('share')